In [1]:
from ultralytics import YOLO

# Load a YOLOv8 pretrained model
model = YOLO('yolov8n.pt')  # or yolov8s.pt for better accuracy

# If you want to fine-tune on VCoR, uncomment and adapt below
model.train(data="D:/archive (13)/data.yaml", epochs=50)


Ultralytics 8.3.108  Python-3.11.7 torch-2.6.0+cpu CPU (AMD Ryzen 7 6800H with Radeon Graphics)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=D:/archive (13)/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True,

RuntimeError: Dataset 'D://archive (13)/data.yaml' error  'D:/archive (13)/data.yaml' does not exist

In [ ]:
import cv2
from ultralytics import YOLO
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk

# Load model
model = YOLO("yolov8n.pt")  # pretrained for car and person detection

# Define color classifier (mockup or integrate ML model)
def classify_car_color(image):
    # This is a placeholder: use color histogram or a trained model for real use
    avg_color = image.mean(axis=0).mean(axis=0)
    b, g, r = avg_color
    if b > 100 and b > r and b > g:
        return "blue"
    return "other"

# Run detection and annotate frame
def process_image(image_path):
    img = cv2.imread(image_path)
    results = model(img)[0]

    person_count = 0

    for box in results.boxes:
        cls_id = int(box.cls[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if cls_id == 0:  # person
            person_count += 1
            label = "Person"
            color = (0, 255, 255)
        elif cls_id in [2, 3, 5, 7]:  # car, motorcycle, bus, truck
            cropped = img[y1:y2, x1:x2]
            car_color = classify_car_color(cropped)
            label = f"{car_color.capitalize()} Car"
            color = (0, 0, 255) if car_color == "blue" else (255, 0, 0)
        else:
            continue

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.putText(img, f"People Count: {person_count}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
    return img

# GUI setup
def open_file():
    filepath = filedialog.askopenfilename()
    if not filepath:
        return
    img = process_image(filepath)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    img_tk = ImageTk.PhotoImage(img_pil)
    panel.config(image=img_tk)
    panel.image = img_tk

root = tk.Tk()
root.title("Car Colour and People Detector")
btn = tk.Button(root, text="Select Image", command=open_file)
btn.pack()

panel = tk.Label(root)
panel.pack()

root.mainloop()


In [5]:
import os
import cv2
import torch
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import tkinter as tk
from tkinter import filedialog
from ultralytics import YOLO

# --------------- STEP 1: TRAIN CAR COLOR CLASSIFIER -------------------
VCOR_PATH = r"D:/archive (13)"  # Adjust this if your structure differs

# Get label names
LABELS = os.listdir(VCOR_PATH)  # ['black', 'blue', 'red', ...]
LABELS.sort()
label2idx = {label: idx for idx, label in enumerate(LABELS)}
idx2label = {v: k for k, v in label2idx.items()}

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load pretrained model and modify for classification
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = torch.nn.Linear(model.fc.in_features, len(LABELS))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Prepare dataset manually
def load_images_and_labels(dataset_path):
    images, labels = [], []
    for label in os.listdir(dataset_path):
        label_path = os.path.join(dataset_path, label)
        if not os.path.isdir(label_path):
            continue
        for img_name in os.listdir(label_path):
            img_path = os.path.join(label_path, img_name)
            try:
                img = Image.open(img_path).convert('RGB')
                img = transform(img)
                images.append(img)
                labels.append(label2idx[label])
            except:
                continue
    return torch.stack(images), torch.tensor(labels)

print("📦 Loading car images for training...")
X_train, y_train = load_images_and_labels(VCOR_PATH)

# Training
print("🧠 Training color classifier...")
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

epochs = 5
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train.to(device))
    loss = criterion(outputs, y_train.to(device))
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

model.eval()
print("✅ Training complete.\n")

# --------------- STEP 2: YOLO FOR PEOPLE DETECTION -------------------
print("📦 Loading YOLOv8 model...")
yolo_model = YOLO('yolov8n.pt')  # uses COCO

# --------------- STEP 3: FUNCTION TO PROCESS IMAGE -------------------
def process_image(img_path):
    image = Image.open(img_path).convert('RGB')
    img_cv = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    annotated_img = img_cv.copy()
    height, width = img_cv.shape[:2]

    # YOLO detection
    results = yolo_model(img_path)[0]
    people_count = 0
    car_boxes = []

    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        class_id = int(class_id)
        if class_id == 0:  # person
            people_count += 1
            cv2.rectangle(annotated_img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.putText(annotated_img, "Person", (int(x1), int(y1)-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        elif class_id == 2:  # car
            car_boxes.append([int(x1), int(y1), int(x2), int(y2)])

    # Car color classification
    car_count = 0
    for (x1, y1, x2, y2) in car_boxes:
        car_crop = Image.fromarray(img_cv[y1:y2, x1:x2])
        car_tensor = transform(car_crop).unsqueeze(0).to(device)
        pred = model(car_tensor).argmax(dim=1).item()
        color = idx2label[pred]
        car_count += 1

        # Rectangle based on color
        if color == 'blue':
            rect_color = (0, 0, 255)  # Red for blue car
        else:
            rect_color = (255, 0, 0)  # Blue for others

        cv2.rectangle(annotated_img, (x1, y1), (x2, y2), rect_color, 2)
        cv2.putText(annotated_img, color, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, rect_color, 2)

    # Display counts
    cv2.putText(annotated_img, f"Cars: {car_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 255, 50), 2)
    cv2.putText(annotated_img, f"People: {people_count}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 255, 50), 2)

    cv2.imshow("Result", cv2.cvtColor(annotated_img, cv2.COLOR_RGB2BGR))
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# --------------- STEP 4: GUI TO UPLOAD IMAGE -------------------
def launch_gui():
    root = tk.Tk()
    root.withdraw()
    file_path = filedialog.askopenfilename(title="Select an image", filetypes=[("Image files", "*.jpg *.jpeg *.png")])
    if file_path:
        process_image(file_path)

# -------------------- START GUI -----------------------
if __name__ == "__main__":
    launch_gui()


📦 Loading car images for training...


RuntimeError: stack expects a non-empty TensorList

In [8]:
import os
from PIL import Image

dataset_path = r"D:/archive (13)/train"  # or the correct path
count = 0
for label in os.listdir(dataset_path):
    label_path = os.path.join(dataset_path, label)
    if not os.path.isdir(label_path):
        continue
    for img_file in os.listdir(label_path):
        img_path = os.path.join(label_path, img_file)
        try:
            with Image.open(img_path) as img:
                count += 1
        except:
            print(f"❌ Failed to open image: {img_path}")

print(f"✅ Total images successfully opened: {count}")


✅ Total images successfully opened: 7267


In [ ]:
import os
import cv2
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from ultralytics import YOLO
import matplotlib.pyplot as plt

# ------------------ CONFIG ------------------
VCOR_PATH = r"D:/archive (13)/train"  # ✅ Your working dataset path
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5

# ------------ Transform -------------
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

# ------------ Load Data -------------
LABELS = sorted(os.listdir(VCOR_PATH))
label2idx = {label: i for i, label in enumerate(LABELS)}
idx2label = {i: label for label, i in label2idx.items()}

def load_images_and_labels(path):
    images, labels = [], []
    for label in os.listdir(path):
        label_path = os.path.join(path, label)
        for img_file in os.listdir(label_path):
            img_path = os.path.join(label_path, img_file)
            try:
                img = Image.open(img_path).convert("RGB")
                img = transform(img)
                images.append(img)
                labels.append(label2idx[label])
            except:
                continue
    return torch.stack(images), torch.tensor(labels)

print("📦 Loading VCoR dataset...")
X, y = load_images_and_labels(VCOR_PATH)
print(f"✅ Loaded {len(X)} images.")

# -------------- Train Classifier --------------
print("🧠 Training Color Classifier...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(LABELS))
model = model.to(DEVICE)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

X, y = X.to(DEVICE), y.to(DEVICE)

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = loss_fn(outputs, y)
    loss.backward()
    optimizer.step()
    acc = (outputs.argmax(1) == y).float().mean()
    print(f"📊 Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.4f} | Accuracy: {acc.item()*100:.2f}%")

# ------------- Save Classifier --------------
torch.save(model.state_dict(), "car_color_model.pth")

# ------------- Load YOLO for detection -------------
print("📦 Loading YOLOv5 for object detection...")
yolo_model = YOLO("yolov5s.pt")

# ------------- Run Detection + Color -------------
def predict_car_color(crop):
    crop = transform(crop.resize((IMG_SIZE, IMG_SIZE))).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output = model(crop)
    pred_idx = output.argmax(1).item()
    return idx2label[pred_idx]

def process_image(image_path):
    image = cv2.imread(image_path)
    results = yolo_model(image)[0]

    people_count = 0

    for box in results.boxes:
        cls_id = int(box.cls.item())
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = results.names[cls_id]

        if label == 'car':
            car_crop = Image.fromarray(image[y1:y2, x1:x2])
            pred_color = predict_car_color(car_crop)

            if pred_color == "blue":
                color = (0, 0, 255)  # 🔴 Red for blue cars
            else:
                color = (255, 0, 0)  # 🔵 Blue for others

            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            cv2.putText(image, pred_color, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        elif label == 'person':
            people_count += 1

    cv2.putText(image, f"People: {people_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    return image

# ------------- Simple GUI -------------
def gui():
    while True:
        img_path = input("\n📸 Enter image path (or type 'exit'): ").strip()
        if img_path.lower() == 'exit':
            break
        if not os.path.exists(img_path):
            print("❌ File not found.")
            continue
        result_img = process_image(img_path)
        cv2.imshow("Car Colour Detection", result_img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

# ------------- Run GUI -------------
gui()


📦 Loading VCoR dataset...
✅ Loaded 7267 images.
🧠 Training Color Classifier...
